In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
# Créer les dossiers pour le projet
!mkdir -p /content/drive/MyDrive/tp4_dasd/data
!mkdir -p /content/drive/MyDrive/tp4_dasd/outputs/stage1
!mkdir -p /content/drive/MyDrive/tp4_dasd/outputs/stage2
!mkdir -p /content/drive/MyDrive/tp4_dasd/configs

In [ ]:
%cd /content/drive/MyDrive/tp4_dasd/configs/
%rm -rf LLaMA-Factory
!git clone --depth 1 https://github.com/hiyouga/LLaMA-Factory.git
%cd LLaMA-Factory
%ls
%pip install -e .[torch,bitsandbytes]
%pip install -U bitsandbytes>=0.46.1

In [ ]:
import json
import os

# Define the directory path
data_dir = "/content/drive/MyDrive/tp4_dasd/data"

# Ensure the directory exists
os.makedirs(data_dir, exist_ok=True)

# Define the content for dataset_info.json
# We map 'prompt' to 'instruction', 'query' to 'input', and 'response' to 'output'
# which is the standard Alpaca format you requested.
dataset_info_content = {
  "dasd_stage1": {
    "file_name": "stage1.json"
  },
  "dasd_stage2": {
    "file_name": "stage2.json"
  }
}

# Write the file
file_path = os.path.join(data_dir, "dataset_info.json")

with open(file_path, 'w') as f:
    json.dump(dataset_info_content, f, indent=2)

print(f"Successfully generated dataset_info.json at: {file_path}")

In [ ]:
%cd /content/drive/MyDrive/tp4_dasd/configs/LLaMA-Factory/

!!llamafactory-cli train \
    --stage sft \
    --do_train \
    --model_name_or_path unsloth/Qwen3-4B-Instruct-2507-unsloth-bnb-4bit \
    --dataset dasd_stage2 \
    --dataset_dir /content/drive/MyDrive/tp4_dasd/configs/LLaMA-Factory/data/ \
    --template qwen \
    --finetuning_type lora \
    --lora_rank 8 \
    --lora_target all \
    --output_dir /content/drive/MyDrive/tp4_dasd/outputs/stage2 \
    --overwrite_output_dir \
    --plot_loss \
    --trust_remote_code \
    --per_device_train_batch_size 1 \
    --gradient_accumulation_steps 8 \
    --learning_rate 2.0e-4 \
    --num_train_epochs 3.0 \
    --lr_scheduler_type cosine \
    --warmup_ratio 0.1 \
    --weight_decay 0.01 \
    --logging_steps 10 \
    --save_steps 500 \
    --cutoff_len 3000 \
    --max_samples 1000 \
    --packing false \
    --preprocessing_num_workers 4 \
    --dataloader_num_workers 2 \
    --fp16 \
    --gradient_checkpointing \
    --optim adamw_bnb_8bit \
    --report_to none

In [ ]:
import torch
try:
  assert torch.cuda.is_available() is True
except AssertionError:
  print("Please set up a GPU before using LLaMA Factory: https://medium.com/mlearning-ai/training-yolov4-on-google-colab-316f8fff99c6")

In [ ]:
import json
import numpy as np
import openai
from concurrent.futures import ThreadPoolExecutor, as_completed

# Configuration API (gardée ici pour clonage local par thread)
API_BASE_URL = "https://api.infomaniak.com/2/ai/48/openai/v1"
API_KEY = "nKuJabWS1epvq3x-m8by6NOU4xP4_znNL9OhmgXBPz9OeWOHlyGJIENnG8oXLT-4oOXNmESqExEMZv6o"
TEACHER_MODEL = "openai/gpt-oss-120b"
BLOC_SIZE = 50
TOTAL_THREADS = 10
LOW_TEMP = 0.15
HIGH_TEMP = 0.9

# Résultats globaux
responses = []

# Chargement des données une seule fois (mémoire)
with open("/content/drive/MyDrive/tp4_dasd/data/result.json", "r") as f:
    data = json.load(f)

def process_block(block_idx, start_idx, end_idx, temperature):
    """Traite un bloc de puzzles (start_idx..end_idx-1) avec la température fournie."""
    local_responses = []
    # Client par thread pour éviter des problèmes de concurrence
    client = openai.OpenAI(base_url=API_BASE_URL, api_key=API_KEY)
    for i in range(start_idx, end_idx):
        try:
            puzzle = data[i]
            puzzle_id = puzzle["puzzle_id"]
            puzzle_initial_fen = puzzle["initial_fen"]
            puzzle_ennemy_moves = puzzle["solution_san"][1::2]
            puzzle_nb_moves = len(puzzle_ennemy_moves) + 1
            print(f"Block {block_idx} - Puzzle {i} [ID:{puzzle_id}] - FEN: {puzzle_initial_fen} - N: {puzzle_nb_moves}")

            response = client.chat.completions.create(
                model=TEACHER_MODEL,
                logprobs=True,
                temperature=temperature,
                messages=[
                    {"role": "system", "content": "You are a chess expert and you'll try to answer chess puzzles. You'll be provided with a position in SAN format and should find the best N moves from that position. N is provided by the user."},
                    {"role": "user", "content": "I have a chess puzzle for you. The initial position is: " + puzzle_initial_fen + ". Try to find a mate or stalemate in " + str(puzzle_nb_moves) + " moves in SAN format. The ennemy will do these moves: " + "/".join(puzzle_ennemy_moves) + ". Return only the moves in SAN format, without any explanation or commentary. If you can't find a solution, return 'No solution found'."},
                ],
                max_tokens=8192,
            )

            # Extraction des logprobs (si fournis)
            logprobs_data = response.choices[0].logprobs
            tokens = []
            logprobs = []
            if logprobs_data:
                for token_info in logprobs_data.content:
                    tokens.append(token_info.token)
                    logprobs.append(token_info.logprob)

            total_logprob = sum(logprobs) if logprobs else 0.0
            mean_logprob = np.exp(np.mean(logprobs)) if logprobs else 0.0

            local_responses.append({
                "puzzle": puzzle_id,
                "content": response.choices[0].message.content,
                "log_prob": mean_logprob,
                "block": block_idx,
								"temperature": temperature
            })
        except Exception as e:
            print(f"Erreur bloc {block_idx} puzzle {i}: {e}")
    return local_responses

blocks = []
for b in range(TOTAL_THREADS):
    start = b * BLOC_SIZE
    end = start + BLOC_SIZE
    temp = LOW_TEMP if b < (TOTAL_THREADS // 2) else HIGH_TEMP
    blocks.append((b, start, end, temp))

# Exécution parallèle (ThreadPoolExecutor convient pour les appels I/O réseau)
with ThreadPoolExecutor(max_workers=TOTAL_THREADS) as ex:
    futures = [ex.submit(process_block, b, s, e, t) for (b,s,e,t) in blocks]
    for fut in as_completed(futures):
        try:
            res = fut.result()
            responses.extend(res)
        except Exception as e:
            print("Un bloc a échoué :", e)

# Enregistrement des réponses agrégées
with open("/content/drive/MyDrive/tp4_dasd/outputs/stage1/responses.json", "w") as f:
    json.dump(responses, f, indent=4)

In [ ]:
import json

%cd /content/LLaMA-Factory/

NAME = "Qwen3"
AUTHOR = "LLaMA Factory"

with open("data/identity.json", "r", encoding="utf-8") as f:
  dataset = json.load(f)

for sample in dataset:
  sample["output"] = sample["output"].replace("{{"+ "name" + "}}", NAME).replace("{{"+ "author" + "}}", AUTHOR)

with open("data/identity.json", "w", encoding="utf-8") as f:
  json.dump(dataset, f, indent=2, ensure_ascii=False)

In [ ]:
%cd /content/LLaMA-Factory/

!llamafactory-cli train \
    --stage sft \
    --do_train \
    --model_name_or_path unsloth/Qwen3-4B-Instruct-2507-unsloth-bnb-4bit \
    --dataset  \
    --template qwen3_nothink \
    --finetuning_type lora \
    --lora_rank 8 \
    --lora_target all \
    --output_dir /content/drive/MyDrive/tp4_dasd/outputs/stage1 \
    --overwrite_output_dir \
    --plot_loss \
    --trust_remote_code \
    --per_device_train_batch_size 1 \
    --gradient_accumulation_steps 8 \
    --learning_rate 1.0e-4 \
    --num_train_epochs 3.0 \
    --lr_scheduler_type cosine \
    --warmup_ratio 0.1 \
    --logging_steps 10 \
    --save_steps 500 \
    --cutoff_len 2048 \
    --max_samples 1000 \
    --preprocessing_num_workers 16 \
    --dataloader_num_workers 4 \
    --fp16 \
    --report_to none